# Desempenho de alunos — análise exploratória

**Status:** Rascunho com exemplo reproduzível.

Explorar notas, dados ausentes e a diferença entre aprovação informada e uma regra simulada. A amostra vem dos dados já versionados no projeto; os nomes foram substituídos por identificadores. Sua origem externa ainda precisa ser documentada.

[README](README.md) · [Catálogo](../../../CATALOGO.md)

## 1. Dados e ambiente

O CSV tem identificador, idade, nota e aprovação informada. Não interpretar aprovação como algo calculado automaticamente pela nota: o campo é preservado da base. As notas ausentes permanecem ausentes.

No Colab, envie amostra_estudo.csv pelo painel de arquivos. No computador, abra este notebook a partir da pasta do projeto ou da raiz do repositório.

In [ ]:
from pathlib import Path
import pandas as pd

candidatos = [
    Path("data/raw/amostra_estudo.csv"),
    Path("amostra_estudo.csv"),
    Path("em-andamento/desempenho-alunos/analise-alunos-1jgyrf/data/raw/amostra_estudo.csv"),
]
caminho = next((p for p in candidatos if p.exists()), None)
if caminho is None:
    raise FileNotFoundError("Obtenha data/raw/amostra_estudo.csv nesta pasta e ajuste o caminho ou envie o arquivo ao Colab.")
dados = pd.read_csv(caminho)
dados.head()

## 2. Qualidade dos dados

Separar contagem de registros, notas observadas e notas ausentes evita transformar falta de informação em desempenho zero.

In [ ]:
resumo = pd.Series({
    "registros": len(dados),
    "notas_observadas": int(dados["nota"].notna().sum()),
    "notas_ausentes": int(dados["nota"].isna().sum()),
    "media_observada": dados["nota"].mean(),
    "mediana_observada": dados["nota"].median(),
    "menor_nota": dados["nota"].min(),
    "maior_nota": dados["nota"].max(),
    "idade_maxima": dados["idade"].max(),
    "aprovados_informados": int(dados["aprovado_informado"].sum()),
    "taxa_aprovacao_informada": dados["aprovado_informado"].mean(),
})
resumo

## 3. Maiores notas

O ranking considera somente notas observadas. Identificadores não representam nomes de estudantes.

In [ ]:
dados.dropna(subset=["nota"]).nlargest(5, "nota")[["aluno_id", "nota"]]

## 4. Cenário didático de bônus

Simular bônus de 40%, limitado a 10, e aprovação por nota final maior ou igual a 6. Esta regra é apenas um exercício, separado do campo de aprovação informado. Não é uma regra escolar inferida da base.

In [ ]:
cenario = dados.copy()
cenario["nota_com_bonus"] = (cenario["nota"] * 1.4).clip(upper=10)
cenario["aprovado_cenario"] = cenario["nota_com_bonus"].ge(6).astype("boolean")
cenario.loc[cenario["nota"].isna(), "aprovado_cenario"] = pd.NA
cenario[["aluno_id", "nota", "nota_com_bonus", "aprovado_informado", "aprovado_cenario"]]

## 5. Associação entre idade e nota

Calcular apenas nas observações disponíveis. A amostra pequena e sem contexto não permite conclusões causais.

In [ ]:
correlacao = dados[["idade", "nota"]].corr().loc["idade", "nota"]
print(f"Correlação de Pearson na amostra observada: {correlacao:.3f}")

## 6. Respostas calculadas

Os números são gerados pelas células; não há conclusões pré-preenchidas.

In [ ]:
print(f"Registros: {len(dados)}; notas ausentes: {dados.nota.isna().sum()}.")
print(f"Notas observadas: mínimo {dados.nota.min():.1f}; máximo {dados.nota.max():.1f}; média {dados.nota.mean():.2f}.")
print(f"Aprovação informada: {dados.aprovado_informado.sum()} de {len(dados)} ({dados.aprovado_informado.mean():.1%}).")

## Próximos passos

- [ ] Documentar origem e licença da amostra.
- [ ] Acrescentar visualizações e discutir impacto dos dados ausentes.
- [ ] Conferir a regra original de aprovação antes de comparar com o cenário.

A organização substituiu o exemplo antigo que preenchia notas ausentes com zero e usava conclusões com X/Y/Z. O código e os dados anteriores permanecem no histórico do Git e no Drive.